<a href="https://colab.research.google.com/github/pcg1974/ebook2audiobook/blob/main/Notebooks/colab_ebook2audiobook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Welcome to the ebook2audiobook Google Colab!
## Features
- 🔧 **TTS Engines supported**: XTTSv2, Bark, Fairseq, VITS, Tacotron2, Tortoise, GlowTTS, YourTTS- 📚 **Convert multiple file formats**: .epub, .mobi, .azw3, .fb2, .lrf, .rb, .snb, .tcr, .pdf, .txt, .rtf, .doc, .docx, .html, .odt, .azw, .tiff, .tif, .png, .jpg, .jpeg, .bmp
- 🔍 **OCR scanning** for files with text pages as images
- 🔊 **High-quality text-to-speech** from near realtime to near real voice
- 🗣️ **Optional voice cloning** using your own voice file
- 🌐 **Supports 1158 languages** ([supported languages list](https://dl.fbaipublicfiles.com/mms/tts/all-tts-languages.html))
- 💻 **Low-resource friendly** — runs on **2 GB RAM / 1 GB VRAM (minimum)**
- 🎵 **Audiobook output formats**: mono or stereo aac, flac, mp3, m4b, m4a, mp4, mov, ogg, wav, webm
- 🧠 **SML tags supported** — fine-grained control of breaks, pauses, voice switching and more ([see below](#sml-tags-available))
- 🧩 **Optional custom model** using your own trained model (XTTSv2 only, other on request)
- 🎛️ **Fine-tuned preset models** trained by the E2A Team<br/>
     <i>(Contact us if you need additional fine-tuned models, or if you'd like to share yours to the official preset list)</i>
## Want to run locally for free? ⬇
## [Check out the ebook2audiobook github!](https://github.com/pcg1974/ebook2audiobook)

In [1]:
# @title 🔒 Minimal-Permission Google Drive Persistence Setup

import os
from google.colab import auth, drive

print("1. Authenticating with restricted drive.file scope...")
# Authenticate using ONLY access to files created or opened by this notebook
auth.authenticate_user(scopes=['https://www.googleapis.com/auth/drive.file'])

print("2. Mounting Google Drive...")
drive.mount('/content/drive', force_remount=True)

# Define permanent storage paths in your Drive
DRIVE_STORAGE = "/content/drive/MyDrive/Colab/ebook2audiobook"
os.makedirs(f"{DRIVE_STORAGE}/voices", exist_ok=True)
os.makedirs(f"{DRIVE_STORAGE}/audiobooks", exist_ok=True)

# Prepare local directory and create symbolic links
SCRIPT_DIR = "/content/ebook2audiobook"
os.makedirs(SCRIPT_DIR, exist_ok=True)

# Link voices folder
os.system(f"rm -rf {SCRIPT_DIR}/voices && ln -s {DRIVE_STORAGE}/voices {SCRIPT_DIR}/voices")

# Link audiobooks output folder
os.system(f"rm -rf {SCRIPT_DIR}/audiobooks && ln -s {DRIVE_STORAGE}/audiobooks {SCRIPT_DIR}/audiobooks")

print("\n✅ Setup complete! Voices and Audiobooks will automatically save to your Google Drive under:")
print(f"   {DRIVE_STORAGE}")

1. Authenticating with restricted drive.file scope...


TypeError: authenticate_user() got an unexpected keyword argument 'scopes'

In [5]:
# @title 🚀 Run ebook2audiobook!

import os
import subprocess
import time
import shutil
import sysconfig
import sys

# Emojis for logs
CHECK_MARK = "✅"
CROSS_MARK = "❌"

SCRIPT_DIR = "/content/ebook2audiobook"
VENV_DIR = f"{SCRIPT_DIR}/python_env"
VENV_PYTHON = f"{VENV_DIR}/bin/python"

# ── Ensure system temporary directory exists BEFORE anything else runs ─────────
os.makedirs(f"{SCRIPT_DIR}/tmp", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)

# ── Environment variables ────────────────────────────────────────────────────
os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"
os.environ["TTS_CACHE"] = f"{SCRIPT_DIR}/models"
os.environ["TESSDATA_PREFIX"] = f"{SCRIPT_DIR}/models/tessdata"
os.environ["TMPDIR"] = "/tmp"  # Use standard /tmp for OS installers like Rustup
os.environ["SCRIPT_MODE"] = "native"  # Direct app.py to run in native environment mode

# PG: Setting this because Colab by default sets it to "module://matplotlib_inline.backend_inline".
os.environ["MPLBACKEND"] = "Agg"

def display_loading_bar(total_steps):
    print("\n--- LOADING... Total steps:", total_steps, " ---")

def update_progress(step, total_steps):
    bar_length = 20
    progress_percent = int((step / total_steps) * 100)
    progress_filled = int(bar_length * step / total_steps)
    bar = '=' * progress_filled + '>' + ' ' * max(bar_length - progress_filled - 1, 0)
    print(f"--- PROGRESS: [{bar}] {progress_percent}% ({step}/{total_steps}) ---")

def run_command_with_log(command, description, step_progress, total_step_commands, forced_cwd=None):
    """Runs a shell command and logs progress, outcome and duration."""
    print(f"\n{step_progress}/{total_step_commands}: {description}...")
    start_time = time.time()
    try:
        if forced_cwd:
            working_dir = forced_cwd
        elif os.path.exists(SCRIPT_DIR):
            working_dir = SCRIPT_DIR
        else:
            working_dir = "/content"

        process = subprocess.Popen(command, shell=True, cwd=working_dir,
                                   stdout=subprocess.PIPE, stderr=subprocess.PIPE)
        stdout, stderr = process.communicate()
        duration = f"{time.time() - start_time:.2f}"
        if process.returncode != 0:
            print(f"{CROSS_MARK} Command failed: {description} (Took {duration}s)")
            print(f"   Command: {command}")
            print(f"   Error Output:\n{stderr.decode()}")
            return False
        else:
            update_progress(step_progress, total_step_commands)
            print(f"{CHECK_MARK} {step_progress}/{total_step_commands} completed: {description} (Took {duration}s)")
            return True
    except Exception as e:
        duration = f"{time.time() - start_time:.2f}"
        print(f"{CROSS_MARK} Error during: {description} (Took {duration}s) — {e}")
        return False


# ── Step 1 : OS-level packages ────────────────────────────────────────────────
os_install_commands = [
    ("apt-get update -qq",
     "Update package lists"),
    ("sudo add-apt-repository ppa:deadsnakes/ppa -y && apt-get update -qq",
     "Add PPA for older Python versions"),
    ("apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils",
     "Install Python 3.10 (Required by ebook2audiobook)"),
    ("apt-get install -y -qq libxcb-cursor0 libegl1 libopengl0",
     "Install Calibre display libraries"),
    ("sudo -v && wget -nv -O- https://download.calibre-ebook.com/linux-installer.sh | sudo sh /dev/stdin",
     "Download & install Calibre"),
    ("apt-get install -y -qq ffmpeg",
     "Install ffmpeg"),
    ("apt-get install -y -qq mediainfo",
     "Install mediainfo"),
    ("apt-get install -y -qq nodejs",
     "Install nodejs"),
    ("apt-get install -y -qq espeak-ng",
     "Install espeak-ng"),
    ("apt-get install -y -qq sox",
     "Install sox"),
    ("apt-get install -y -qq tesseract-ocr tesseract-ocr-eng",
     "Install Tesseract OCR + English language pack"),
    ("apt-get install -y -qq mecab libmecab-dev mecab-ipadic-utf8",
     "Install mecab (Japanese text analysis)"),
    ("curl -fsSL https://sh.rustup.rs | sh -s -- -y --quiet && "
     "echo 'source $HOME/.cargo/env' >> ~/.bashrc",
     "Install Rust (required by some Python packages)"),
]

# ── Step 2 : Git clone ────────────────────────────────────────────────────────
git_commands = [
    (f"rm -rf {SCRIPT_DIR} && git clone --depth=1 https://github.com/pcg1974/ebook2audiobook.git {SCRIPT_DIR}",
     "Git clone ebook2audiobook"),
]

# ── Step 3 : Virtual Environment & Python packages ───────────────────────────
pip_commands = [
    (f"python3.10 -m venv --without-pip {VENV_DIR}",
     "Create Python 3.10 virtual environment in python_env"),
    (f"curl -sSL https://bootstrap.pypa.io/get-pip.py | {VENV_PYTHON}",
     "Bootstrap pip inside virtual environment"),
    (f"{VENV_PYTHON} -m pip install -q --upgrade pip setuptools wheel packaging",
     "Upgrade pip / setuptools / wheel inside venv"),
    (f"{VENV_PYTHON} -m pip install -q --upgrade llvmlite numba --only-binary=:all:",
     "Install llvmlite & numba"),
    (f"{VENV_PYTHON} -m pip install -q 'unidic==1.1.0'",
     "Install pinned unidic 1.1.0 package"),
    (f"{VENV_PYTHON} -m pip install -e {SCRIPT_DIR}/ext/py/demucs --no-deps -q",
     "Install local demucs package"),
    (f"{VENV_PYTHON} -m pip install -q --no-cache-dir -r {SCRIPT_DIR}/requirements.txt",
     "Install Python requirements from requirements.txt"),
    (f"{VENV_PYTHON} -m unidic download",
     "Download Unidic dictionary data"),
]


# ── Helper: sitecustomize hook ───────────────────────────────────────────────
def install_sitecustomize():
    src = f"{SCRIPT_DIR}/components/sitecustomize.py"
    dst_dir = subprocess.check_output(
        [VENV_PYTHON, "-c", "import sysconfig; print(sysconfig.get_paths()['purelib'])"]
    ).decode().strip()

    dst = os.path.join(dst_dir, "sitecustomize.py")
    if not os.path.exists(src):
        print(f"{CROSS_MARK} sitecustomize.py source not found at {src}, skipping.")
        return False
    if not os.path.exists(dst) or os.path.getmtime(src) > os.path.getmtime(dst):
        shutil.copy2(src, dst)
        print(f"{CHECK_MARK} Installed sitecustomize.py hook → {dst}")
    else:
        print(f"{CHECK_MARK} sitecustomize.py already up-to-date.")
    return True


# ════════════════════════════════════════════════════════════════════════════
# EXECUTION
# ════════════════════════════════════════════════════════════════════════════

os.chdir("/content")

# --- Step 1: OS packages ---
print("\n--- Step 1: OS-Level Installations ---")
print("Installs system packages required by the bash script. (~3-5 min)")
display_loading_bar(len(os_install_commands))
step1_ok = True
for i, (cmd, desc) in enumerate(os_install_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(os_install_commands), forced_cwd="/content"):
        step1_ok = False
        break

cargo_env = os.path.expanduser("~/.cargo/env")
if os.path.exists(cargo_env):
    os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ.get("PATH", "")

if step1_ok:
    print(f"\n{CHECK_MARK} Step 1: OS-Level Installations — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 1: OS-Level Installations — Warning: Step 1 had issues.")


# --- Step 2: Git Clone ---
print("\n--- Step 2: Git Clone ---")
display_loading_bar(len(git_commands))
step2_ok = True
for i, (cmd, desc) in enumerate(git_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(git_commands), forced_cwd="/content"):
        step2_ok = False
        break

if step2_ok:
    print(f"\n{CHECK_MARK} Step 2: Git Clone — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 2: Git Clone — Failed. Aborting script.")
    sys.exit(1)


# --- Step 3: Python Virtual Environment & Package Installation ---
print("\n--- Step 3: Virtual Environment Setup & Package Installation ---")
print("Creates python_env and installs Python requirements. (~3-5 min)")
display_loading_bar(len(pip_commands))
step3_ok = True
for i, (cmd, desc) in enumerate(pip_commands):
    if not run_command_with_log(cmd, desc, i + 1, len(pip_commands)):
        step3_ok = False
        break

if step3_ok:
    print(f"\n{CHECK_MARK} Step 3: Python Package Installation — Completed Successfully!")
else:
    print(f"\n{CROSS_MARK} Step 3: Python Package Installation — Failed. See errors above.")
    sys.exit(1)


# --- Step 4: Installing hooks and patches ---
print("\n--- Step 4: Installing sitecustomize.py hook & applying unidic patches ---")
install_sitecustomize()

# 1. Direct source patch for unidic.__init__ (exposes DICDIR to fugashi/XTTS)
unidic_init = f"{VENV_DIR}/lib/python3.10/site-packages/unidic/__init__.py"
if os.path.exists(unidic_init):
    with open(unidic_init, "r", encoding="utf-8") as f:
        content = f.read()
    if "DICDIR =" not in content:
        with open(unidic_init, "a", encoding="utf-8") as f:
            f.write("\n\n# Patch for XTTS / fugashi compatibility\nDICDIR = dicdir\n")
        print(f"{CHECK_MARK} Patched unidic/__init__.py for DICDIR compatibility.")
    else:
        print(f"{CHECK_MARK} unidic/__init__.py already contains DICDIR patch.")

# 2. Append fallback DICDIR check to virtual environment sitecustomize.py
sitecustomize_path = f"{VENV_DIR}/lib/python3.10/site-packages/sitecustomize.py"
patch_code = """
try:
    import unidic
    if not hasattr(unidic, 'DICDIR'):
        unidic.DICDIR = getattr(unidic, 'dicdir', '')
except Exception:
    pass
"""
with open(sitecustomize_path, "a", encoding="utf-8") as f:
    f.write(patch_code)
print(f"{CHECK_MARK} Configured runtime sitecustomize.py fallback hook.")


# --- Step 5: Create required directories & missing config files ---
print("\n--- Step 5: Creating required directories & mecabrc config ---")
for d in ["models", "models/tessdata", "tmp", "run", "audiobooks", "ebooks", "voices"]:
    os.makedirs(f"{SCRIPT_DIR}/{d}", exist_ok=True)
os.chmod(f"{SCRIPT_DIR}/tmp", 0o777)

# Create the missing mecabrc file inside UniDic dicdir
unidic_dicdir = f"{VENV_DIR}/lib/python3.10/site-packages/unidic/dicdir"
os.makedirs(unidic_dicdir, exist_ok=True)
mecabrc_path = os.path.join(unidic_dicdir, "mecabrc")
if not os.path.exists(mecabrc_path):
    with open(mecabrc_path, "w", encoding="utf-8") as f:
        f.write("# empty mecabrc file for unidic\n")
    print(f"{CHECK_MARK} Created missing mecabrc file in UniDic dicdir.")

print(f"{CHECK_MARK} Directories and config files ready.")


# --- Step 5.5: Precise literal patch ---
print("\n--- Step 5.5: Patching ebook2audiobook source bug ---")
patch_file = f"{SCRIPT_DIR}/lib/classes/device_installer.py"
if os.path.exists(patch_file):
    with open(patch_file, "r", encoding="utf-8") as f:
        code = f.read()

    # 1. Fix the original UnboundLocalError bypass
    code = code.replace(
        "tag_ver = _normalize_version(ver_str)",
        "tag_ver = __import__('packaging.version').version.parse(str(ver_str))"
    )

    # 2. Fix the newly discovered TypeError on line 851
    # (Forces the string `version` to become a Version object so it matches `v`)
    code = code.replace(
        "v <= version",
        "v <= __import__('packaging.version').version.parse(str(version))"
    )

    with open(patch_file, "w", encoding="utf-8") as f:
        f.write(code)
    print(f"{CHECK_MARK} Cleanly patched device_installer.py bugs (both Scope and Type errors).")


# --- Step 6: Run App ---
print("\n--- Step 6: Launch ebook2audiobook ---")
print("Starting the Gradio web interface with a public share link...")
try:
    get_ipython().system(
        f"cd {SCRIPT_DIR} && "
        f"VIRTUAL_ENV={VENV_DIR} {VENV_PYTHON} -u app.py --script_mode native --share"
    )
except Exception as e:
    print(f"{CROSS_MARK} Error starting app.py: {e}")

print("\n--- All Steps Completed ---")
print("Check the output above for the public Gradio URL.")


--- Step 1: OS-Level Installations ---
Installs system packages required by the bash script. (~3-5 min)

--- LOADING... Total steps: 13  ---

1/13: Update package lists...
--- PROGRESS: [=>                  ] 7% (1/13) ---
✅ 1/13 completed: Update package lists (Took 8.08s)

2/13: Add PPA for older Python versions...
--- PROGRESS: [===>                ] 15% (2/13) ---
✅ 2/13 completed: Add PPA for older Python versions (Took 13.83s)

3/13: Install Python 3.10 (Required by ebook2audiobook)...
--- PROGRESS: [====>               ] 23% (3/13) ---
✅ 3/13 completed: Install Python 3.10 (Required by ebook2audiobook) (Took 16.44s)

4/13: Install Calibre display libraries...
--- PROGRESS: [======>             ] 30% (4/13) ---
✅ 4/13 completed: Install Calibre display libraries (Took 6.47s)

5/13: Download & install Calibre...
--- PROGRESS: [=======>            ] 38% (5/13) ---
✅ 5/13 completed: Download & install Calibre (Took 48.28s)

6/13: Install ffmpeg...
--- PROGRESS: [=========>         

In [ ]:
# Launch app directly using the existing virtual environment
!cd /content/ebook2audiobook && \
 VIRTUAL_ENV=/content/ebook2audiobook/python_env \
 /content/ebook2audiobook/python_env/bin/python -u app.py --script_mode native --share

v26.8.20 native mode
---> Hardware detected: {'name': 'cuda', 'os': 'manylinux_2_28', 'arch': 'x86_64', 'pyvenv': [3, 10], 'tag': 'cu128', 'note': ''}
IPs available for connection:
['127.0.0.1', '::1', '172.28.0.12']
Note: 0.0.0.0 is not the IP to connect. Instead use an IP above to connect and port 7860
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://0db0998168f6cc3e88.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
 Note: access limit time: 3 days. Your browser needs cookies enabled to resume the conversions.
Processing eBook file: charles-dickens_the-mystery-of-edwin-drood_advanced.epub
vram_dict: {'os': 'linux', 'device_type': 'cuda', 'device_name': 'Tesla T4', 'free_bytes': 15527116800, 'total_bytes': 15637086208, 'allocated_bytes': 0, 'reserved_bytes': 0, 'free_vram_gb': 15, 'tota

In [6]:
import os

# Create the missing mecabrc file inside unidic
unidic_dir = "/content/ebook2audiobook/python_env/lib/python3.10/site-packages/unidic/dicdir"
os.makedirs(unidic_dir, exist_ok=True)

mecabrc_path = os.path.join(unidic_dir, "mecabrc")
with open(mecabrc_path, "w") as f:
    f.write("# empty mecabrc file for unidic\n")

print("✅ Created missing mecabrc file.")

✅ Created missing mecabrc file.


In [9]:
import os

sitecustomize_path = "/content/ebook2audiobook/python_env/lib/python3.10/site-packages/sitecustomize.py"

# Patch code that forces unidic to expose DICDIR when app.py runs
patch_code = """
try:
    import unidic
    if not hasattr(unidic, 'DICDIR'):
        unidic.DICDIR = getattr(unidic, 'dicdir', '')
except Exception:
    pass
"""

with open(sitecustomize_path, "a", encoding="utf-8") as f:
    f.write(patch_code)

print("✅ Patched unidic.DICDIR directly inside virtual environment!")

✅ Patched unidic.DICDIR directly inside virtual environment!


In [1]:
!/content/ebook2audiobook/python_env/bin/python -m pip install -q "unidic==1.1.0"
!/content/ebook2audiobook/python_env/bin/python -m unidic download

/bin/bash: line 1: /content/ebook2audiobook/python_env/bin/python: No such file or directory
/bin/bash: line 1: /content/ebook2audiobook/python_env/bin/python: No such file or directory
